# 07 · Evaluate All Three Models — the deck's accuracy slide

> **OWNER:** Either member — run this LAST, after both tracks (M1's 01-02,
> M4's 03-06) are done.
> **PREREQUISITES:** `models/yolo_rdd.pt`, `models/yolo_hazard.pt`,
> `models/yolo_plate.pt` all present. Each check below tells you which is
> missing if you run this early — that's fine, it's meant to be re-run.
> **EXPECTED RUNTIME:** ~20 minutes.
> **OUTPUTS:** a combined per-class metrics table, a combined latency table
> (including the all-three-models-on-one-frame cost), one annotated video
> over 3 test clips, a markdown + LaTeX table for the deck, and
> `models/MODEL_CARDS.md` consolidating all three cards.

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 0 — load all three models (skips gracefully if one isn't trained yet)

In [ ]:
from ultralytics import YOLO

from common import constants

MODEL_PATHS = {key: MODEL_ROOT / filename for key, filename in constants.MODEL_FILES.items()}
models = {}
for key, path in MODEL_PATHS.items():
    if path.exists():
        models[key] = YOLO(str(path))
        print(f"loaded {key}: {path}")
    else:
        print(f"MISSING {key}: {path} not found — run its training notebook first (02 / 04 / 06-or-05's pretrained path).")

if len(models) < 3:
    print()
    print(f"only {len(models)}/3 models present — the sections below run against whatever IS present")
    print("and skip the rest. Re-run this notebook once all three exist for the full deck output.")

## Step 1 — combined per-class metrics table

In [ ]:
import pandas as pd

from common import evaluate

DATA_YAMLS = {
    "rdd": DATA_ROOT / "rdd2022_india" / "data.yaml",
    "hazard": DATA_ROOT / "hazards_prepared" / "data.yaml",
    "plate": DATA_ROOT / "plates_prepared" / "data.yaml",
}
CLASS_NAME_SETS = {"rdd": constants.RDD_DETECTION_NAMES, "hazard": constants.HAZARD_CLASSES, "plate": constants.PLATE_CLASSES}

combined_tables = []
for key, model in models.items():
    data_yaml_path = DATA_YAMLS[key]
    if not data_yaml_path.exists():
        print(f"skipping {key}: {data_yaml_path} not found (was a pretrained plate model used instead of 06? that path has no local data.yaml)")
        continue
    table = evaluate.per_class_table(model, str(data_yaml_path), split="test")
    table.insert(0, "model", key)
    combined_tables.append(table)

if combined_tables:
    combined_metrics = pd.concat(combined_tables, ignore_index=True)
    print(combined_metrics.to_string(index=False))
else:
    combined_metrics = pd.DataFrame()
    print("no models had a local data.yaml to evaluate against (common if plates used a pretrained model with no local test split)")

## Step 2 — combined latency table

Per-model latency across backends, plus the **combined per-frame cost if all
three run on one frame** — this is the number behind any edge-feasibility
claim, so it is reported honestly, including when a backend is unavailable
rather than silently omitted.

In [ ]:
from common import export

latency_by_model = {}
for key, model in models.items():
    pt_path = MODEL_PATHS[key]
    onnx_path = pt_path.with_suffix(".onnx")
    if not onnx_path.exists():
        print(f"skipping {key}: {onnx_path} not found — export it in that model's training notebook first")
        continue
    print(f"benchmarking {key}...")
    latency_by_model[key] = export.benchmark_latency(pt_path, onnx_path, imgsz=640)
    export.print_latency_table(latency_by_model[key])
    print()

print("=" * 60)
print("COMBINED per-frame cost if all three models run on one frame:")
for backend_key in ("torch_gpu_ms", "torch_cpu_ms", "onnx_cpu_ms"):
    values = [latency_by_model[k].get(backend_key) for k in latency_by_model]
    if any(v is None for v in values) or not values:
        print(f"  {backend_key}: n/a (not every model has this backend's number)")
    else:
        total_ms = sum(values)
        print(f"  {backend_key}: {total_ms:.2f} ms total  ({1000 / total_ms:.1f} fps if run sequentially on one frame)")
print("=" * 60)

## Step 3 — full stack on 3 test clips, one annotated video

Runs all available models over three short clips and overlays every
detection on one output video — the thing you actually play in the pitch.
Point `TEST_CLIPS` at real files before running; this notebook does not
fabricate video.

In [ ]:
import cv2

TEST_CLIPS = sorted((REPO_ROOT / "data" / "raw_video").glob("*.mp4"))[:3]
OUTPUT_VIDEO_PATH = MODEL_ROOT / "combined_demo_annotated.mp4"

if not TEST_CLIPS:
    print(f"no test clips found under {REPO_ROOT / 'data' / 'raw_video'} — drop up to 3 short .mp4 clips there and re-run.")
elif not models:
    print("no models loaded — nothing to overlay. Run Step 0 once at least one model exists.")
else:
    writer = None
    total_frames = 0
    for clip_path in TEST_CLIPS:
        cap = cv2.VideoCapture(str(clip_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 15.0
        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if writer is None:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(str(OUTPUT_VIDEO_PATH), fourcc, fps, (w, h))
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            annotated = frame
            for key, model in models.items():
                annotated = model.predict(annotated, verbose=False)[0].plot()
            writer.write(annotated)
            total_frames += 1
        cap.release()
        print(f"processed {clip_path.name}")
    if writer is not None:
        writer.release()
    print(f"wrote {total_frames} annotated frames -> {OUTPUT_VIDEO_PATH}")

## Step 4 — deck-ready tables: markdown + LaTeX

In [ ]:
if not combined_metrics.empty:
    markdown_table = combined_metrics.to_markdown(index=False)
    latex_table = combined_metrics.to_latex(index=False, float_format="%.3f")

    (MODEL_ROOT / "deck_metrics_table.md").write_text(markdown_table)
    (MODEL_ROOT / "deck_metrics_table.tex").write_text(latex_table)

    print("markdown table (paste into the deck / README):")
    print(markdown_table)
    print()
    print(f"wrote models/deck_metrics_table.md and models/deck_metrics_table.tex")
else:
    print("no combined metrics available yet — see Step 1")

## Step 5 — consolidated MODEL_CARDS.md

In [ ]:
card_paths = [MODEL_ROOT / name for name in ("MODEL_CARD_rdd.md", "MODEL_CARD_hazard.md", "MODEL_CARD_plate.md")]
found_cards = [p for p in card_paths if p.exists()]
missing_cards = [p for p in card_paths if not p.exists()]

if missing_cards:
    print(f"missing: {[p.name for p in missing_cards]} — those training notebooks haven't produced a card yet.")

consolidated = "\n\n---\n\n".join(p.read_text() for p in found_cards)
(MODEL_ROOT / "MODEL_CARDS.md").write_text(consolidated)
print(f"consolidated {len(found_cards)}/3 cards -> models/MODEL_CARDS.md")

---
### What this notebook produced
- A combined per-class metrics table (`models/deck_metrics_table.md` / `.tex`)
- A combined latency table, including the all-three-on-one-frame edge-feasibility number
- `models/combined_demo_annotated.mp4` — the pitch video
- `models/MODEL_CARDS.md` — all three cards consolidated

### Next
Nothing — this is the end of the ML track. Paste `deck_metrics_table.md` into
the deck and hand `combined_demo_annotated.mp4` to whoever is presenting.